# Machine Fault Recognition

## Imports

In [1]:
import os, re
import glob
import numpy as np
import pandas as pd
from collections import defaultdict
from joblib import Parallel, delayed

import joblib

import librosa
import librosa.display 
import IPython.display as ipd

import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from tqdm.auto import tqdm

from itertools import cycle

sns.set_theme(style="white", palette=None)
color_pal = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])

"""
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
"""


# !pip install wandb

import wandb
# wandb.login()

## Preprocessing Helper Functions 

In [2]:
def create_resampler(orig_sr: int, new_sr: int):
    resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=new_sr)
    return resampler

def resample(audio: torch.Tensor, resampler) -> torch.Tensor:
    audio = resampler(audio)
    return audio


def peak_normalize(audio: torch.Tensor) -> torch.Tensor: 
    max_val = torch.max(torch.abs(audio))

    if max_val == 0:
        return audio
    
    return audio / max_val


def rms_normalize(audio: torch.Tensor) -> torch.Tensor: 
    rms = torch.sqrt(torch.mean(audio * audio))
    return audio / (rms + 1e-8)


def cut(audio: torch.Tensor, sr: int, lower_cut=0.5, upper_cut=9.5) -> torch.Tensor:
    start = int(lower_cut * sr)
    end = int(upper_cut * sr)
    audio = audio[start:end]
    return audio

In [10]:
# Data Deduplication

TARGET_SR = 16000

def extract_rich_feature(path: str, n_mfcc: int = 40) -> torch.Tensor:
    audio, sr = torchaudio.load(path)

    audio = audio.mean(dim=0, keepdim=True)  # [1, T]

    if sr != TARGET_SR:
        audio = torchaudio.transforms.Resample(sr, TARGET_SR)(audio)

     # Normalize amplitude to remove volume-based false uniqueness
    audio = audio / (audio.abs().max() + 1e-8)

    mfcc_transform = torchaudio.transforms.MFCC(
        sample_rate=TARGET_SR,
        n_mfcc=n_mfcc,
        melkwargs={
            "n_fft":       1024,
            "hop_length":  256,   # finer time resolution than before
            "n_mels":      64,
            "f_min":       50.0,
            "f_max":       8000.0,
        }
    )

    mfcc = mfcc_transform(audio).squeeze(0)  # [n_mfcc, T]

    # Statistics across time — captures SHAPE of the signal, not just mean
    feat_mean = mfcc.mean(dim=1)              # [n_mfcc]
    feat_std  = mfcc.std(dim=1)               # [n_mfcc]
    feat_max  = mfcc.max(dim=1).values        # [n_mfcc]
    
    # Delta MFCCs (first-order temporal derivative) — catches rhythmic patterns
    delta = mfcc[:, 1:] - mfcc[:, :-1]       # [n_mfcc, T-1]
    feat_delta_mean = delta.mean(dim=1)       # [n_mfcc]
    feat_delta_std  = delta.std(dim=1)        # [n_mfcc]

    return torch.cat([
        feat_mean, feat_std, feat_max,
        feat_delta_mean, feat_delta_std
    ])  # [n_mfcc * 5] = 200-dim


def calibrate_threshold(
    file_paths: list[str],
    labels: list[int],
    sample_size: int = 200,
    device: str = "cuda"
):
    """
    Samples files and prints similarity distribution to help you
    pick the right threshold — run this once before deduplication.
    """
    import random
    sample = random.sample(list(zip(file_paths, labels)), min(sample_size, len(file_paths)))

    print("Extracting sample features for calibration...")
    feats = torch.stack([extract_rich_feature(p) for p, _ in tqdm(sample)])
    feats = F.normalize(feats.to(device), dim=1)
    sim   = torch.matmul(feats, feats.T)

    mask     = ~torch.eye(len(sample), dtype=torch.bool, device=device)
    off_diag = sim[mask]

    print(f"\n📊 Similarity Distribution (n={len(sample)} files)")
    print(f"  Min:    {off_diag.min():.4f}")
    print(f"  Mean:   {off_diag.mean():.4f}")
    print(f"  Median: {off_diag.median():.4f}")
    print(f"  Max:    {off_diag.max():.4f}")
    print(f"  Std:    {off_diag.std():.4f}")
    print()
    for t in [0.90, 0.93, 0.95, 0.97, 0.98, 0.99, 0.995, 0.999]:
        pct = (off_diag > t).float().mean().item() * 100
        print(f"  Pairs > {t}: {pct:5.1f}%  ← {'⚠️ too aggressive' if pct > 30 else '✅ reasonable' if pct < 5 else '🔶 check'}")

    print("\n💡 Recommendation: pick threshold where < 5% of pairs match")

def extract_wrapper(path):
    return extract_feature_torch(path)

def get_machine_group(path: str) -> str:
    match = re.search(r'[Mm]achine[\s_-]?(\d+)', path)
    return f"Machine {match.group(1)}" if match else "Unknown"

def _load_or_extract(cache_key: str, paths: list[str], device: str) -> torch.Tensor:
    feat_file     = f"{cache_key}_features.pt"
    manifest_file = f"{cache_key}_manifest.txt"

    if os.path.exists(feat_file) and os.path.exists(manifest_file):
        with open(manifest_file) as f:
            cached = f.read().splitlines()
        if cached == paths:
            print(f"  ⚡ Cache hit: {cache_key}")
            return torch.load(feat_file, weights_only=True).to(device)
        else:
            # Debug: show WHY cache missed
            print(f"  ⚠️  Cache stale: {cache_key}")
            print(f"      Cached paths:  {len(cached)}")
            print(f"      Current paths: {len(paths)}")
            if len(cached) == len(paths):
                # Same length but different content — find first mismatch
                for i, (c, p) in enumerate(zip(cached, paths)):
                    if c != p:
                        print(f"      First mismatch at index {i}:")
                        print(f"        cached:  {c}")
                        print(f"        current: {p}")
                        break

    print(f"  🔧 Extracting: {cache_key}")

    # ── Clean single progress bar ──────────────────────────────────────────
    with tqdm(total=len(paths), desc="  Extracting", unit="file", dynamic_ncols=True) as pbar:
        def extract_and_update(p):
            result = extract_rich_feature(p)
            pbar.update(1)
            return result

        feats = Parallel(n_jobs=4, prefer="threads")(  # threads share pbar cleanly
            delayed(extract_and_update)(p) for p in paths
        )
    # ───────────────────────────────────────────────────────────────────────

    feats = torch.stack(feats)
    torch.save(feats, feat_file)
    with open(manifest_file, "w") as f:
        f.write("\n".join(paths))

    return feats.to(device)
    
def deduplicate_fast(
    file_paths: list[str],
    labels: list[int],
    threshold: float = 0.999,    # ← start high, calibrate downward
    device: str = "cuda"
) -> tuple[list[str], list[int]]:
    """
    Deduplication per (machine, label) group using rich 200-dim MFCC features.
    Always run calibrate_threshold() first to verify your threshold.
    """
    groups: dict[tuple, list] = defaultdict(list)
    for path, label in zip(file_paths, labels):
        groups[(get_machine_group(path), label)].append((path, label))

    final_files, final_labels = [], []
    total_before = total_after = 0

    for (machine, label), items in sorted(groups.items()):
        paths = [p for p, _ in items]
        labs  = [l for _, l in items]
        print(f"\n🚀 {machine} | Label {label} ({len(paths)} files)")

        cache_key = f"{machine.replace(' ', '_')}_label{label}"
        feats = _load_or_extract(cache_key, paths, device)
        feats = F.normalize(feats, dim=1)

        sim_matrix = torch.matmul(feats, feats.T)

        keep = torch.ones(len(paths), dtype=torch.bool, device=device)
        for i in tqdm(range(len(paths)), desc="  Deduplicating"):
            if not keep[i]:
                continue
            dupes    = sim_matrix[i] > threshold
            dupes[i] = False
            keep[dupes] = False
            
        kept    = keep.sum().item()
        removed = len(paths) - kept
        total_before += len(paths)
        total_after  += kept
        print(f"  📊 {len(paths)} → {kept} (removed {removed})")

        keep_cpu = keep.cpu()
        for i in range(len(paths)):
            if keep_cpu[i]:
                final_files.append(paths[i])
                final_labels.append(labs[i])

    print(f"\n{'='*50}")
    print(f"✅ Total: {total_before} → {total_after} (removed {total_before - total_after})")
    return final_files, final_labels

In [24]:
## Data Augmentation

# 1. Adding noise
# Adding gaussian noise
# Start with SNR = 15–30 dB
# Lower = noisier
def add_noise_snr(audio: torch.Tensor, snr_db_range=(15, 35)) -> torch.Tensor:
    # Randomly samples a single SNR value (in decibels) uniformly from the given range. Higher = cleaner signal.
    snr_db = torch.FloatTensor(1).uniform_(*snr_db_range).item()
    
    # Creates white Gaussian noise with the same shape and device as the audio.
    noise = torch.randn_like(audio)

    # Computes mean power (mean of squared amplitudes) for both. .clamp(min=1e-8) prevents division by zero on silent audio
    signal_power = audio.pow(2).mean().clamp(min=1e-8)  # avoid div/0
    noise_power  = noise.pow(2).mean().clamp(min=1e-8)

    # Derives a scalar so that after multiplying noise by it, signal_power / (scale² × noise_power) = 10^(snr_db/10).
    # This is the standard SNR formula rearranged
    scale = torch.sqrt(signal_power / (10 ** (snr_db / 10) * noise_power))
    
    return audio + scale * noise

# 2. Time Shift
# Preserves structure
# Simulates recording misalignment
# Circularly shifts the audio waveform in time, wrapping with zeros (not circular — it drops samples and pads with silence).
def time_shift(audio: torch.Tensor, sr: int, max_shift_sec: float = 0.1) -> torch.Tensor:
    max_shift = int(max_shift_sec * sr)
    # Converts max shift from seconds to samples (e.g. 0.1s × 16000 = 1600 samples).
  
    shift = torch.randint(-max_shift, max_shift + 1, (1,)).item()
    # Randomly picks an integer shift, which can be positive (shift right) or negative (shift left).

    if shift > 0:
        # Shift right: prepend shift zeros, drop the last shift samples.
        return torch.cat([torch.zeros(shift, device=audio.device), audio[:-shift]])
    elif shift < 0:
        # Shift left: drop the first |shift| samples, append |shift| zeros.
        return torch.cat([audio[-shift:], torch.zeros(-shift, device=audio.device)])

    return audio

# 3. Time strectching and compressing
# Machines run at slightly different speeds
# VERY useful for generalization
# rate can be >1 or <1
# Speeds up or slows down the audio without changing pitch (uses phase-vocoder under the hood via librosa)
def time_stretch(audio: torch.Tensor, rate_range=(0.9, 1.1)) -> torch.Tensor:
    rate     = torch.FloatTensor(1).uniform_(*rate_range).item()
    audio_np = audio.squeeze().cpu().numpy()  # ensure 1D
    stretched = librosa.effects.time_stretch(y=audio_np, rate=rate)
    return torch.tensor(stretched, dtype=audio.dtype).to(audio.device)


# 4. Random cropping
def random_crop(audio: torch.Tensor, sr: int, duration_sec: float = 9.0) -> torch.Tensor:
    target_len = int(sr * duration_sec)
    T = audio.shape[-1]  # works for both [T] and [1, T]
    if T < target_len:
        # Pad with zeros rather than returning a short clip
        pad = target_len - T
        return torch.nn.functional.pad(audio, (0, pad))
    if T == target_len:
        return audio
    max_start = T - target_len
    start = torch.randint(0, max_start + 1, (1,)).item()
    return audio[..., start:start + target_len]  # ... handles both [T] and [1,T]


# 5. Apply this after extracting spectrograms
# Works best with log-Mel spectrograms
# Applies frequency masking first (zeros out random horizontal bands in the mel-frequency axis),
# then time masking (zeros out random vertical bands in the time axis). 
# Both are torchaudio transforms that operate on [..., freq, time] tensors. 
# Applying both together is the SpecAugment policy from the original paper
# valid
def apply_specaugment(
    spec: torch.Tensor,
    freq_masker: T.FrequencyMasking,
    time_masker: T.TimeMasking
) -> torch.Tensor:
    return time_masker(freq_masker(spec))

In [1]:
## MFCC Augmentations
# ─── 1. SpecAugment on MFCC ───────────────────────────────────────────────────
# Same concept as mel spectrogram, but parameters must be smaller.
# freq_mask_param should be at most n_mfcc // 4  (e.g. 4 for 13 coeffs, 10 for 40)
# time_mask_param stays the same logic (depends on time axis length)

def build_mfcc_augmenters(n_mfcc: int = 13, time_mask_param: int = 30):
    """
    Build SpecAugment transforms sized appropriately for MFCC dimensions.
    Call once at init, reuse per sample — same pattern as the dataset class.
    """
    freq_mask_param = max(1, n_mfcc // 4)   # e.g. 3 for n_mfcc=13, 10 for n_mfcc=40
    freq_masker = T.FrequencyMasking(freq_mask_param=freq_mask_param)
    time_masker = T.TimeMasking(time_mask_param=time_mask_param)
    return freq_masker, time_masker


#valid
def apply_mfcc_specaugment(
    mfcc: torch.Tensor,
    freq_masker: T.FrequencyMasking,
    time_masker: T.TimeMasking,
) -> torch.Tensor:
    """
    Apply SpecAugment to an MFCC tensor.
    Input shape: [n_mfcc, time] or [1, n_mfcc, time]
    Masks random cepstral coefficient bands and random time frames.
    """
    return time_masker(freq_masker(mfcc))


# ─── 2. Cepstral Coefficient Noise ────────────────────────────────────────────
# Adds small Gaussian noise directly to MFCC values.
# Simulates sensor variability and slight environmental differences.
# Unlike waveform noise (which affects all coefficients uniformly after DCT),
# this targets the cepstral domain directly.
# std=0.1–0.5 is a reasonable range for normalized MFCCs.

#valid
def add_cepstral_noise(
    mfcc: torch.Tensor,
    std_range: tuple[float, float] = (0.05, 0.2),
) -> torch.Tensor:
    """
    Add Gaussian noise directly to MFCC coefficients.
    Input shape: [n_mfcc, time] or [1, n_mfcc, time]
    """
    std = torch.FloatTensor(1).uniform_(*std_range).item()
    noise = torch.randn_like(mfcc) * std
    return mfcc + noise


# ─── 3. High Coefficient Dropout ──────────────────────────────────────────────
# Zeros out the top-k cepstral coefficients (the high-frequency detail ones).
# Simulates losing fine spectral texture — like low-bitrate recording or
# a machine running at slightly different speed.
# Specifically meaningful for MFCCs because lower coefficients = coarse spectral
# shape (more stable), higher = fine detail (more noise-sensitive).

#valid
def drop_high_coefficients(
    mfcc: torch.Tensor,
    drop_fraction: float = 0.3,
) -> torch.Tensor:
    """
    Zero out the top `drop_fraction` of cepstral coefficients.
    Input shape: [n_mfcc, time] or [1, n_mfcc, time]
    E.g. drop_fraction=0.3 on 13 coeffs → zeros out coefficients 10,11,12
    """
    n_mfcc = mfcc.shape[-2]                         # works for both [C,T] and [1,C,T]
    n_drop = max(1, int(n_mfcc * drop_fraction))
    result = mfcc.clone()
    result[..., -n_drop:, :] = 0.0                  # zero the highest coefficients
    return result


# ─── 4. Temporal Delta Jitter ─────────────────────────────────────────────────
# MFCCs are often used with delta (velocity) and delta-delta (acceleration)
# features appended. If you use those, adding small noise to the delta channels
# only (not C0) simulates temporal uncertainty.
# Only relevant if your extract_mfcc returns [3*n_mfcc, time] with deltas stacked.

def jitter_delta_channels(
    mfcc_with_deltas: torch.Tensor,
    n_mfcc: int,
    std: float = 0.05,
) -> torch.Tensor:
    """
    Add small noise only to the delta and delta-delta channels, not the static ones.
    Assumes stacked layout: [static | delta | delta-delta] along the coefficient axis.
    Input shape: [3*n_mfcc, time]
    """
    result = mfcc_with_deltas.clone()
    result[n_mfcc:, :] += torch.randn_like(result[n_mfcc:, :]) * std
    return result

NameError: name 'torch' is not defined

## Feature Extraction Helper Functions

In [8]:

def extract_mfcc(audio: torch.Tensor, sr, n_mfcc=13):
    mfcc_transform = torchaudio.transforms.MFCC(sample_rate=sr, n_mfcc=n_mfcc,
        melkwargs={
            "n_fft": 1024,
            "hop_length": 512,
            "n_mels": 40
        }
    )

    return mfcc_transform(audio)


def mfcc_stats(mfcc):

    mean = mfcc.mean(dim=1)
    std = mfcc.std(dim=1)

    return torch.cat([mean, std])


def mfcc_stats_with_deltas(mfcc):

    delta = F.compute_deltas(mfcc)
    delta2 = F.compute_deltas(delta)

    return torch.cat([
        mfcc.mean(dim=1), mfcc.std(dim=1),
        delta.mean(dim=1), delta.std(dim=1),
        delta2.mean(dim=1), delta2.std(dim=1)
    ])


def extract_spectrogram(audio: torch.Tensor, n_fft=1024, hop_length=512):
    # audio shape must be [1, T]
    if audio.dim() == 1:
        audio = audio.unsqueeze(0)

    transform = torchaudio.transforms.Spectrogram(
        n_fft=n_fft,
        hop_length=hop_length,
        power=2.0
    )

    S = transform(audio)  # [1, freq_bins, time]

    S_db = torchaudio.transforms.AmplitudeToDB()(S)

    return S_db.squeeze(0)  # [freq_bins, time]    


def extract_mel_spectrogram(audio: torch.Tensor, sr: int,
                                   n_fft=1024, hop_length=512, n_mels=40):

    if audio.dim() == 1:
        audio = audio.unsqueeze(0)

    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0
    )

    mel = mel_transform(audio)

    mel_db = torchaudio.transforms.AmplitudeToDB()(mel)

    return mel_db.squeeze(0)  # [n_mels, time]

## Data Loading

Machine 1 + Normal     → 0

Machine 1 + Abnormal   → 1

Machine 2 + Normal     → 2

Machine 2 + Abnormal   → 3

Machine 3 + Normal     → 4

Machine 3 + Abnormal   → 5

In [6]:
DATA_PATH = "/kaggle/input/datasets/mmagdy908/machine-audio-for-pattern-recognition"

label_map = {
    ("Machine 1", "Normal"): 0,
    ("Machine 1", "Abnormal"): 1,
    ("Machine 2", "Normal"): 2,
    ("Machine 2", "Abnormal"): 3,
    ("Machine 3", "Normal"): 4,
    ("Machine 3", "Abnormal"): 5,
}

file_paths = []
labels = []

for machine in ["Machine 1", "Machine 2", "Machine 3"]:
    for condition in ["Normal", "Abnormal"]:
        
        folder = os.path.join(
            DATA_PATH,
            machine,
            "machine_data",
            condition
        )
        
        files = glob.glob(os.path.join(folder, "*.wav"))
        
        for f in files:
            file_paths.append(f)
            labels.append(label_map[(machine, condition)])

In [11]:
import torch
import torch.nn.functional as F

# Take a small sample - 50 files from Machine 1
sample_paths = [p for p in file_paths if "Machine 1" in p][:50]
feats = torch.stack([extract_rich_feature(p) for p in sample_paths])
feats = F.normalize(feats, dim=1)
sim_matrix = torch.matmul(feats, feats.T)

# Look at the OFF-diagonal similarities (exclude self-similarity)
mask = ~torch.eye(50, dtype=torch.bool)
off_diag = sim_matrix[mask]

print(f"Min similarity:    {off_diag.min():.4f}")
print(f"Max similarity:    {off_diag.max():.4f}")
print(f"Mean similarity:   {off_diag.mean():.4f}")
print(f"Median similarity: {off_diag.median():.4f}")

# Distribution
thresholds = [0.90, 0.92, 0.95, 0.97, 0.98, 0.99, 0.999]
for t in thresholds:
    pct = (off_diag > t).float().mean().item() * 100
    print(f"  > {t}: {pct:.1f}% of pairs")

Min similarity:    0.8168
Max similarity:    0.9990
Mean similarity:   0.9609
Median similarity: 0.9720
  > 0.9: 93.1% of pairs
  > 0.92: 86.7% of pairs
  > 0.95: 70.5% of pairs
  > 0.97: 51.7% of pairs
  > 0.98: 38.7% of pairs
  > 0.99: 21.8% of pairs
  > 0.999: 0.0% of pairs


### Confirm Data Loading

In [12]:
print(f"Total files: {len(file_paths)}")

pd.Series(labels).value_counts()

Total files: 56236


0    16200
2    16200
4    14400
3     3240
1     3176
5     3020
Name: count, dtype: int64

In [13]:
# ── Step 1: Calibrate FIRST (takes ~1 min on 200 samples) ──
calibrate_threshold(file_paths, labels, sample_size=200, device="cuda")



Extracting sample features for calibration...


  0%|          | 0/200 [00:00<?, ?it/s]


📊 Similarity Distribution (n=200 files)
  Min:    0.1567
  Mean:   0.8151
  Median: 0.8802
  Max:    0.9990
  Std:    0.1734

  Pairs > 0.9:  46.3%  ← ⚠️ too aggressive
  Pairs > 0.93:  37.7%  ← ⚠️ too aggressive
  Pairs > 0.95:  29.1%  ← 🔶 check
  Pairs > 0.97:  16.8%  ← 🔶 check
  Pairs > 0.98:   9.2%  ← 🔶 check
  Pairs > 0.99:   4.0%  ← ✅ reasonable
  Pairs > 0.995:   1.9%  ← ✅ reasonable
  Pairs > 0.999:   0.0%  ← ✅ reasonable

💡 Recommendation: pick threshold where < 5% of pairs match


In [14]:
clean_files, clean_labels = deduplicate_fast(
    file_paths, labels,
    threshold=0.999,   # ← replace with calibrated value
    device="cuda"
)


🚀 Machine 1 | Label 0 (16200 files)
  🔧 Extracting: Machine_1_label0


  Extracting:   0%|          | 0/16200 [00:00<?, ?file/s]

  Deduplicating:   0%|          | 0/16200 [00:00<?, ?it/s]

  📊 16200 → 8664 (removed 7536)

🚀 Machine 1 | Label 1 (3176 files)
  🔧 Extracting: Machine_1_label1


  Extracting:   0%|          | 0/3176 [00:00<?, ?file/s]

...

  Deduplicating:   0%|          | 0/14400 [00:00<?, ?it/s]

  📊 14400 → 9446 (removed 4954)

🚀 Machine 3 | Label 5 (3020 files)
  🔧 Extracting: Machine_3_label5


  Extracting:   0%|          | 0/3020 [00:00<?, ?file/s]

  Deduplicating:   0%|          | 0/3020 [00:00<?, ?it/s]

  📊 3020 → 2994 (removed 26)

✅ Total: 56236 → 41870 (removed 14366)


In [16]:
print(f"Total files: {len(clean_files)}")

pd.Series(clean_labels).value_counts()

Total files: 41870


2    14466
4     9446
0     8664
3     3223
1     3077
5     2994
Name: count, dtype: int64

## Data Splitting

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(clean_files, clean_labels, test_size=0.3, random_state=42, stratify=clean_labels)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(len(X_train), len(X_val), len(X_test))

29309 6280 6281


| Train       | Validation | Test |
|------------|--------|--------|
| 29309       | 6280   | 6281   |


## GPU Setup

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Dataset Class

In [1]:
class MachineAudioDatasetAugmented(Dataset):
    def __init__(
        self,
        file_paths: list[str],
        labels: list[int],
        target_sr: int = 16000,
        train: bool = True,
        duration_sec: float = 9.0,
        n_mfcc: int = 13
    ):
        self.file_paths   = file_paths
        self.labels       = labels
        self.target_sr    = target_sr
        self.train        = train
        self.duration_sec = duration_sec

        # Cache resamplers by source sr so we never rebuild the same one twice
        # e.g. {22050: Resample(22050→16000), 44100: Resample(44100→16000)}
        self._resampler_cache: dict[int, torchaudio.transforms.Resample] = {}

        # SpecAugment transforms — built once, reused every sample
        self.freq_masker = T.FrequencyMasking(freq_mask_param=15)
        self.time_masker = T.TimeMasking(time_mask_param=30)
        
        # SpecAugment for MFCC — smaller params, built separately
        self.mfcc_freq_masker, self.mfcc_time_masker = build_mfcc_augmenters(
            n_mfcc=n_mfcc,
            time_mask_param=30,
        )
        self.n_mfcc = n_mfcc
    # ─────────────────────────────────────────────────────────────────────────
    def __len__(self) -> int:
        return len(self.file_paths)

    # ─────────────────────────────────────────────────────────────────────────
    def _get_resampler(self, orig_sr: int) -> torchaudio.transforms.Resample:
        """Return a cached resampler for this source rate, creating if needed."""
        if orig_sr not in self._resampler_cache:
            self._resampler_cache[orig_sr] = create_resampler(orig_sr, self.target_sr)
        return self._resampler_cache[orig_sr]

    # ─────────────────────────────────────────────────────────────────────────
    def __getitem__(self, idx: int):
        path  = self.file_paths[idx]
        label = self.labels[idx]

        # ── Load ──────────────────────────────────────────────────────────────
        audio, sr = torchaudio.load(path)
        audio = audio.mean(dim=0)           # stereo → mono  [T]

        # ── Resample ──────────────────────────────────────────────────────────
        if sr != self.target_sr:
            resampler = self._get_resampler(sr)
            audio = resample(audio, resampler)

        # ── RMS Normalize ─────────────────────────────────────────────────────
        audio = rms_normalize(audio)

        # ── Crop ──────────────────────────────────────────────────────────────
        if self.train:
            # Random crop → positional variety during training
            audio = random_crop(audio, self.target_sr, self.duration_sec)
        else:
            # Fixed cut (0.5 s → 0.5+duration s) → deterministic for validation
            audio = cut(audio, self.target_sr,
                        lower_cut=0.5,
                        upper_cut=0.5 + self.duration_sec)

        # ── Waveform Augmentation (train only) ────────────────────────────────
        if self.train:
            if torch.rand(1) < 0.5:
                audio = add_noise_snr(audio, snr_db_range=(15, 35))

            if torch.rand(1) < 0.4:
                audio = time_shift(audio, self.target_sr, max_shift_sec=0.1)

            if torch.rand(1) < 0.3:
                audio = time_stretch(audio, rate_range=(0.9, 1.1))
                # ⚠ time_stretch changes length → re-crop to enforce fixed size
                audio = random_crop(audio, self.target_sr, self.duration_sec)

        # ── Mel Spectrogram ───────────────────────────────────────────────────
        mel = extract_mel_spectrogram(audio, self.target_sr)

        # ── SpecAugment (train only) ──────────────────────────────────────────
        if self.train:
            mel = apply_specaugment(mel, self.freq_masker, self.time_masker)

        # ── MFCC path ─────────────────────────────────────────────────────────
        mfcc = extract_mfcc(audio, self.target_sr)          # your existing fn

        if self.train:
            if torch.rand(1) < 0.5:
                mfcc = apply_mfcc_specaugment(
                    mfcc, self.mfcc_freq_masker, self.mfcc_time_masker
                )
            if torch.rand(1) < 0.4:
                mfcc = add_cepstral_noise(mfcc, std_range=(0.05, 0.2))
            if torch.rand(1) < 0.3:
                mfcc = drop_high_coefficients(mfcc, drop_fraction=0.3)

        return mel,mfcc,torch.tensor(label, dtype=torch.long)

NameError: name 'Dataset' is not defined

### Create Dataset

In [ ]:
train_dataset_aug = MachineAudioDatasetAugmented(X_train, y_train)
val_dataset_aug   = MachineAudioDatasetAugmented(X_val, y_val)
test_dataset_aug  = MachineAudioDatasetAugmented(X_test, y_test)

### Create Data Loader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Feature Extraction

In [ ]:
import torch
import torchaudio
import torchaudio.functional as F


class FeatureExtractor:
    def __init__(self, sr=22050, feature_type="mel"):
        self.sr = sr
        self.feature_type = feature_type

        self.mfcc_transform = torchaudio.transforms.MFCC(
            sample_rate=sr,
            n_mfcc=13,
            melkwargs={
                "n_fft": 1024,
                "hop_length": 512,
                "n_mels": 40
            }
        )

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=1024,
            hop_length=512,
            n_mels=40,
            power=2.0
        )

        self.spec_transform = torchaudio.transforms.Spectrogram(
            n_fft=1024,
            hop_length=512,
            power=2.0
        )

        self.db_transform = torchaudio.transforms.AmplitudeToDB()


    def mfcc(self, audio):
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)

        mfcc = self.mfcc_transform(audio)
        return mfcc.squeeze(0)

    def mfcc_stats(self, audio):
        mfcc = self.mfcc(audio)

        mean = mfcc.mean(dim=1)
        std = mfcc.std(dim=1)

        return torch.cat([mean, std])

    def mfcc_stats_with_deltas(self, audio):
        mfcc = self.mfcc(audio)

        delta = F.compute_deltas(mfcc)
        delta2 = F.compute_deltas(delta)

        return torch.cat([
            mfcc.mean(dim=1), mfcc.std(dim=1),
            delta.mean(dim=1), delta.std(dim=1),
            delta2.mean(dim=1), delta2.std(dim=1)
        ])


    def spectrogram(self, audio):
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)

        spec = self.spec_transform(audio)
        spec_db = self.db_transform(spec)

        return spec_db.squeeze(0)


    def mel_spectrogram(self, audio):
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)

        mel = self.mel_transform(audio)
        mel_db = self.db_transform(mel)

        return mel_db.squeeze(0)


    def extract(self, audio):
        if self.feature_type == "mfcc":
            return self.mfcc_stats_with_deltas(audio)

        elif self.feature_type == "mfcc_simple":
            return self.mfcc_stats(audio)

        elif self.feature_type == "mel":
            return self.mel_spectrogram(audio)

        elif self.feature_type == "spec":
            return self.spectrogram(audio)

        else:
            raise ValueError("Unknown feature type")

## Preprocessoring

In [ ]:
class AudioPreprocessor:
    def __init__(self, orig_sr=22050, target_sr=16000, mode="rms"):
        self.target_sr = target_sr
        self.orig_sr = orig_sr
        self.mode = mode
        self.resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=target_sr)

    def resample(self, audio, sr):
        if sr != self.target_sr:
            audio = self.resampler(audio)
        return audio

    def cut(self, audio):
        start = int(0.5 * self.target_sr)
        end = int(9.5 * self.target_sr)
        return audio[:, start:end]

    def normalize(self, audio):
        if self.mode == "peak":
            max_val = torch.max(torch.abs(audio))
            return audio / (max_val + 1e-8)

        elif self.mode == "rms":
            rms = torch.sqrt(torch.mean(audio ** 2))
            return audio / (rms + 1e-8)

        else:
            return audio

## ML Models

### Evaluation Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
import numpy as np


def evaluate_model(model, X, y, num_classes=6):


    y_pred = model.predict(X)


    cm = confusion_matrix(y, y_pred)

    plt.figure(figsize=(6,6))
    sns.heatmap(cm, annot=True, fmt="d")
    plt.title("Confusion Matrix")
    
    wandb.log({"confusion_matrix": wandb.Image(plt)})

    results = {
        "Accuracy": accuracy_score(y, y_pred),


        "Precision_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y, y_pred, average="macro"),
        "F1_macro": f1_score(y, y_pred, average="macro"),

        "Precision_weighted": precision_score(y, y_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y, y_pred, average="weighted"),
        "F1_weighted": f1_score(y, y_pred, average="weighted"),


        "MCC": matthews_corrcoef(y, y_pred)
    }

    return results

### Building Features

In [ ]:
X_train_feats = []
y_train_feats = []

preprocessor = AudioPreprocessor()
extractor = FeatureExtractor(feature_type="mfcc")

for path, label in tqdm(zip(X_train, y_train), total=len(X_train)):
    audio, sr = torchaudio.load(path)
    audio = audio.mean(dim=0)

    # preprocessing
    audio = preprocessor.resample(audio.unsqueeze(0), sr)
    audio = preprocessor.cut(audio)
    audio = preprocessor.normalize(audio)

    # feature extraction
    feat = extractor.extract(audio)

    X_train_feats.append(feat.numpy())
    y_train_feats.append(label)

X_train_feats = np.array(X_train_feats)
y_train_feats = np.array(y_train_feats)

In [ ]:
np.save("X_train_feats.npy", X_train_feats)
np.save("y_train_feats.npy", y_train_feats)

In [ ]:
X_val_feats = []
y_val_feats = []

preprocessor = AudioPreprocessor()
extractor = FeatureExtractor(feature_type="mfcc")

for path, label in tqdm(zip(X_val, y_val), total=len(X_val)):
    audio, sr = torchaudio.load(path)
    audio = audio.mean(dim=0)

    # preprocessing
    audio = preprocessor.resample(audio.unsqueeze(0), sr)
    audio = preprocessor.cut(audio)
    audio = preprocessor.normalize(audio)

    # feature extraction
    feat = extractor.extract(audio)

    X_val_feats.append(feat.numpy())
    y_val_feats.append(label)

X_val_feats = np.array(X_val_feats)
y_val_feats = np.array(y_val_feats)

In [ ]:
np.save("X_val_feats.npy", X_val_feats)
np.save("y_val_feats.npy", y_val_feats)

In [ ]:
X_test_feats = []
y_test_feats = []

preprocessor = AudioPreprocessor()
extractor = FeatureExtractor(feature_type="mfcc")

for path, label in tqdm(zip(X_test, y_test), total=len(X_test)):
    audio, sr = torchaudio.load(path)
    audio = audio.mean(dim=0)

    # preprocessing
    audio = preprocessor.resample(audio.unsqueeze(0), sr)
    audio = preprocessor.cut(audio)
    audio = preprocessor.normalize(audio)

    # feature extraction
    feat = extractor.extract(audio)

    X_test_feats.append(feat.numpy())
    y_test_feats.append(label)

X_test_feats = np.array(X_test_feats)
y_test_feats = np.array(y_test_feats)

In [ ]:
np.save("X_test_feats.npy", X_test_feats)
np.save("y_test_feats.npy", y_test_feats)

### LogisticRegression

In [ ]:
wandb.init(
    project="machine-fault-recognition",
    name="mfcc-lr-baseline",
    config={
        "feature_type": "mfcc_stats_with_deltas",
        "model": "Logistic Regression",
        "n_estimators": 200,
        "sr": 16000,
        "n_fft": 1024,
        "hop_length": 512,
        "n_mels": 40
    }
)

In [ ]:
from sklearn.linear_model import LogisticRegression

LR_model = LogisticRegression(max_iter=2000)
LR_model.fit(X_train_feats, y_train_feats)

results = evaluate_model(LR_model, X_val_feats, y_val_feats)
print(results)
wandb.log(results)

### RandomForestClassifier

In [ ]:
wandb.init(
    project="machine-fault-recognition",
    name="mfcc-rf-baseline",
    config={
        "feature_type": "mfcc_stats_with_deltas",
        "model": "RandomForest",
        "n_estimators": 200,
        "sr": 16000,
        "n_fft": 1024,
        "hop_length": 512,
        "n_mels": 40
    }
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

RF_model = RandomForestClassifier(n_estimators=200)
RF_model.fit(X_train_feats, y_train_feats)

results = evaluate_model(RF_model, X_val_feats, y_val_feats)

print(results)
wandb.log(results)

### 